In [26]:
import psutil 
import platform 
import os
env_name = os.environ.get('CONDA_DEFAULT_ENV')
print("当前 conda 环境名：", env_name)
print(platform.system()) # 操作系统名称 
print(platform.release()) # 操作系统版本 
print(platform.machine()) # 计算机架构 
print(platform.processor()) # 处理器类型 
# CPU 信息 
print(psutil.cpu_count()) # CPU 核数 
print(psutil.cpu_freq()) # CPU 频率 
# 内存信息 
print(psutil.virtual_memory()) # 内存总量、可用内存、已用内存等

当前 conda 环境名： FLLFFL
Windows
11
AMD64
Intel64 Family 6 Model 151 Stepping 2, GenuineIntel
24
scpufreq(current=2000.0, min=0.0, max=2000.0)
svmem(total=34008584192, available=11655090176, percent=65.7, used=22353494016, free=11655090176)


## 2 序列模型

### 2.1 理论计算题

In [2]:
V = {'a', 'b', 'c'}
p_a_given_b = (1 + 1) / (2 + len(V))
p_c_given_b = (1 + 1) / (2 + len(V))
print(f"p('a' | 'b') = {p_a_given_b}")
print(f"p('c' | 'b') = {p_c_given_b}")

p('a' | 'b') = 0.4
p('c' | 'b') = 0.4


### 2.2 编程题

In [3]:
import re
from collections import Counter
def preprocess_text(text, n):
    # 1. 转换为小写，去除标点符号
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    # 2. 按空格分词
    tokens = text.split()
    # 3. 构建词汇表
    word_freq = Counter(tokens)
    sorted_words = sorted(word_freq, key=lambda x: (-word_freq[x], x))
    vocab = {w: i for i, w in enumerate(sorted_words)}
    # 4. 滑动窗口生成特征和标签
    features = []
    labels = []
    for i in range(len(tokens) - n + 1):
        features.append(tokens[i:i+n])
        if i + n < len(tokens):
            labels.append(tokens[i+n])
        else:
            labels.append(None)
    return vocab, (features, labels)

In [4]:
vocab, (features, labels) = preprocess_text("The time machine", n=2)
print("vocab:", vocab)
print("features:", features)
print("labels:", labels)

vocab: {'machine': 0, 'the': 1, 'time': 2}
features: [['the', 'time'], ['time', 'machine']]
labels: ['machine', None]


## 3 循环神经网络

### 3.1 理论计算题

In [5]:
print("dL/dW_hh = sum_{k=1}^{T} (h_T - y) * prod_{j=k}^{T-1} W_hh * h_{k-1}")
print("梯度消失: |W_hh| < 1")
print("梯度爆炸: |W_hh| > 1")

dL/dW_hh = sum_{k=1}^{T} (h_T - y) * prod_{j=k}^{T-1} W_hh * h_{k-1}
梯度消失: |W_hh| < 1
梯度爆炸: |W_hh| > 1


### 3.2 编程题

In [6]:
import numpy as np

In [7]:
def rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h):
    h_t = np.tanh(np.dot(x_t, W_hx.T) + np.dot(h_prev, W_hh.T) + b_h)
    cache = (x_t, h_prev, W_hx, W_hh, h_t)
    return h_t, cache

In [8]:
def rnn_step_backward(dh_next, cache):
    x_t, h_prev, W_hx, W_hh, h_t = cache
    dtanh = dh_next * (1 - h_t ** 2)
    dW_hx = np.dot(dtanh.T, x_t)
    dW_hh = np.dot(dtanh.T, h_prev)
    db_h = np.sum(dtanh, axis=0)
    dx_t = np.dot(dtanh, W_hx)
    dh_prev = np.dot(dtanh, W_hh)
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

In [9]:
batch_size, input_size, hidden_size = 2, 3, 4
x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(hidden_size, input_size)
W_hh = np.random.randn(hidden_size, hidden_size)
b_h = np.random.randn(hidden_size)

h_t, cache = rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h)
print("h_t shape:", h_t.shape)

dh_next = np.random.randn(batch_size, hidden_size)
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_step_backward(dh_next, cache)
print("dx_t shape:", dx_t.shape)
print("dh_prev shape:", dh_prev.shape)
print("dW_hx shape:", dW_hx.shape)
print("dW_hh shape:", dW_hh.shape)
print("db_h shape:", db_h.shape)

h_t shape: (2, 4)
dx_t shape: (2, 3)
dh_prev shape: (2, 4)
dW_hx shape: (4, 3)
dW_hh shape: (4, 4)
db_h shape: (4,)


In [10]:
def numerical_gradient(f, x, h=1e-5):
    fx = f(x)
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        ix = it.multi_index
        oldval = x[ix]
        x[ix] = oldval + h
        fxph = f(x)
        x[ix] = oldval - h
        fxmh = f(x)
        x[ix] = oldval
        grad[ix] = (fxph - fxmh) / (2 * h)
        it.iternext()
    return grad

def loss_W_hh(W):
    h, _ = rnn_step_forward(x_t, h_prev, W_hx, W, b_h)
    return np.sum(h ** 2)

num_grad = numerical_gradient(loss_W_hh, W_hh)
h_t, cache = rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h)
_, _, _, ana_dW_hh, _ = rnn_step_backward(2 * h_t, cache)
print("数值梯度与解析梯度误差:", np.linalg.norm(num_grad - ana_dW_hh))

数值梯度与解析梯度误差: 8.306331490272608e-11


## 4 高级循环神经网络

### 4.1 理论计算题

In [11]:
L, H, D, O = 2, 128, 256, 10
layer1 = 2 * (H*D + H*H + H)
layerN = 2 * (H*2*H + H*H + H)
output = O*(2*H) + O
total = layer1 + (L-1)*layerN + output
print(f"深度双向RNN: L={L}, H={H}, D={D}, O={O}")
print("第1层:", layer1)
print("后续每层:", layerN)
print("输出层:", output)
print("总参数:", total)

深度双向RNN: L=2, H=128, D=256, O=10
第1层: 98560
后续每层: 98560
输出层: 2570
总参数: 199690


### 4.2 编程题

In [12]:
import torch
import torch.nn as nn

In [13]:
def bidirectional_rnn_encoder(X, input_dim, hidden_dim, num_layers=1):
    seq_len, batch_size, _ = X.shape
    rnn = nn.RNN(input_dim, hidden_dim, num_layers,
                 bidirectional=True, batch_first=False)
    outputs, hidden = rnn(X)
    forward_h = hidden[-2]
    backward_h = hidden[-1]
    final_state = torch.cat([forward_h, backward_h], dim=1)
    return outputs, final_state

In [14]:
seq_len, batch, input_dim, hidden_dim = 10, 4, 32, 64
X = torch.randn(seq_len, batch, input_dim)
outputs, final_state = bidirectional_rnn_encoder(X, input_dim, hidden_dim)
print(f"outputs: {outputs.shape}")
print(f"final_state: {final_state.shape}")

outputs: torch.Size([10, 4, 128])
final_state: torch.Size([4, 128])


In [15]:
def manual_bidirectional_rnn(X, W_hx_f, W_hh_f, b_h_f,
                              W_hx_b, W_hh_b, b_h_b):
    seq_len, batch_size, input_dim = X.shape
    hidden_dim = W_hh_f.shape[0]
    # 前向
    h_f = torch.zeros(batch_size, hidden_dim)
    fwd_out = []
    for t in range(seq_len):
        h_f = torch.tanh(X[t] @ W_hx_f.T + h_f @ W_hh_f.T + b_h_f)
        fwd_out.append(h_f)
    # 后向
    h_b = torch.zeros(batch_size, hidden_dim)
    bwd_out = []
    for t in range(seq_len-1, -1, -1):
        h_b = torch.tanh(X[t] @ W_hx_b.T + h_b @ W_hh_b.T + b_h_b)
        bwd_out.insert(0, h_b)
    outputs = torch.stack([torch.cat([f, b], dim=1)
                          for f, b in zip(fwd_out, bwd_out)])
    final_state = torch.cat([fwd_out[-1], bwd_out[0]], dim=1)
    return outputs, final_state

In [16]:
hidden_dim = 64
W_hx_f = torch.randn(hidden_dim, input_dim) * 0.1
W_hh_f = torch.randn(hidden_dim, hidden_dim) * 0.1
b_h_f = torch.zeros(hidden_dim)
W_hx_b = torch.randn(hidden_dim, input_dim) * 0.1
W_hh_b = torch.randn(hidden_dim, hidden_dim) * 0.1
b_h_b = torch.zeros(hidden_dim)
out, final = manual_bidirectional_rnn(X, W_hx_f, W_hh_f, b_h_f,
                                       W_hx_b, W_hh_b, b_h_b)
print(f"out: {out.shape}")
print(f"final: {final.shape}")

out: torch.Size([10, 4, 128])
final: torch.Size([4, 128])


## 5 嵌入向量

### 5.1 理论计算题

In [17]:
print("损失函数: J = -log(sigmoid(v_c . u_o)) - sum_{k=1}^K log(sigmoid(-v_c . u_{n_k}))")
print("负采样: 从 P(w) = freq(w)^0.75 / sum(freq(w')^0.75) 采样")
print("目标: 最大化正样本概率，最小化负样本概率")

损失函数: J = -log(sigmoid(v_c . u_o)) - sum_{k=1}^K log(sigmoid(-v_c . u_{n_k}))
负采样: 从 P(w) = freq(w)^0.75 / sum(freq(w')^0.75) 采样
目标: 最大化正样本概率，最小化负样本概率


### 5.2 编程题

In [18]:
import torch.nn.functional as F

In [19]:
def cbow_forward(context_indices, target_index, W, W_out):
    V, d = W.shape
    # 平均上下文向量
    h = torch.mean(W[context_indices], dim=0)
    # 输出分数
    scores = W_out.T @ h
    # softmax 概率
    probs = F.softmax(scores, dim=0)
    # 交叉熵损失
    loss = -torch.log(probs[target_index])
    return loss, probs

In [20]:
V, d = 100, 16
W = torch.randn(V, d)
W_out = torch.randn(d, V)
context = torch.tensor([1, 3, 5])
target = 7

loss, probs = cbow_forward(context, target, W, W_out)
print(f"loss: {loss.item():.4f}")
print(f"probs shape: {probs.shape}")
print(f"target prob: {probs[target].item():.4f}")

loss: 6.8129
probs shape: torch.Size([100])
target prob: 0.0011


## 6 注意力机制

### 6.1 理论计算题

In [21]:
# Q: (2,4), K: (3,4), V: (3,5), d_k=4
Q = torch.tensor([[1., 0., 1., 0.],
                  [0., 1., 0., 1.]])
K = torch.tensor([[1., 0., 0., 1.],
                  [0., 1., 1., 0.],
                  [1., 1., 0., 0.]])
V = torch.tensor([[1., 2., 3., 4., 5.],
                  [2., 3., 4., 5., 6.],
                  [3., 4., 5., 6., 7.]])
d_k = 4

# Step 1: 得分矩阵 QK^T / sqrt(d_k)
scores = Q @ K.T / (d_k ** 0.5)
print("scores:")
print(scores)

# Step 2: softmax
attn_weights = F.softmax(scores, dim=-1)
print("\nsoftmax:")
print(attn_weights)

# Step 3: 加权求和
output = attn_weights @ V
print("\noutput:")
print(output)

scores:
tensor([[0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000]])

softmax:
tensor([[0.3333, 0.3333, 0.3333],
        [0.3333, 0.3333, 0.3333]])

output:
tensor([[2., 3., 4., 5., 6.],
        [2., 3., 4., 5., 6.]])


### 6.2 编程题

In [22]:
def multi_head_attention(X, W_q, W_k, W_v, W_o, num_heads=2):
    seq_len, batch, d_model = X.shape
    d_k = d_model // num_heads
    
    # 线性投影
    Q = X @ W_q  # (seq_len, batch, d_model)
    K = X @ W_k
    V = X @ W_v
    
    # 分头
    Q = Q.view(seq_len, batch, num_heads, d_k).transpose(0, 2)  # (num_heads, batch, seq_len, d_k)
    K = K.view(seq_len, batch, num_heads, d_k).transpose(0, 2)
    V = V.view(seq_len, batch, num_heads, d_k).transpose(0, 2)
    
    # 缩放点积注意力
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    attn = F.softmax(scores, dim=-1)
    head_output = torch.matmul(attn, V)  # (num_heads, batch, seq_len, d_k)
    
    # 拼接头
    head_output = head_output.transpose(0, 2).contiguous()  # (seq_len, batch, num_heads, d_k)
    concat = head_output.view(seq_len, batch, d_model)
    
    # 最终线性层
    output = concat @ W_o
    return output

In [23]:
seq_len, batch, d_model = 5, 2, 4
num_heads = 2
d_k = d_model // num_heads

X = torch.randn(seq_len, batch, d_model)
W_q = torch.randn(d_model, d_model)
W_k = torch.randn(d_model, d_model)
W_v = torch.randn(d_model, d_model)
W_o = torch.randn(d_model, d_model)

out = multi_head_attention(X, W_q, W_k, W_v, W_o, num_heads)
print(f"input: {X.shape}")
print(f"output: {out.shape}")

input: torch.Size([5, 2, 4])
output: torch.Size([5, 2, 4])
